# 01 — German Credit validation walkthrough (thin companion)

Portfolio companion notebook for the **CLI-first** P0–P3 demo that shipped in PRs #5–#8.

This is **not** the full R-10 §5.1 tutorial rewrite. It wraps the same commands as `README.md` / `docs/dev.md` and ends with the P3 §II side-by-side diff that R-10 called for.

| Surface | Canonical path |
| ------- | -------------- |
| CLI (CI / reproducible) | `workbench validate run --config …` |
| This notebook | Human review + §II overlay diff |

**Prerequisite:** `uv sync --all-extras --dev` from the repo root; Git LFS model present (`git lfs pull`).


## §0 Setup


In [ ]:
from __future__ import annotations

import os
import re
import subprocess
from pathlib import Path

# Resolve repo root whether the kernel cwd is repo root or notebooks/
CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / "pyproject.toml").exists() else CWD.parent
assert (ROOT / "pyproject.toml").exists(), f"Cannot find repo root from {CWD}"
os.chdir(ROOT)
print("repo root:", ROOT)

def run(cmd: list[str]) -> subprocess.CompletedProcess[str]:
    print("$", " ".join(cmd))
    proc = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr)
        raise RuntimeError(f"command failed ({proc.returncode}): {' '.join(cmd)}")
    return proc


## §1 Fixture integrity (optional smoke)


In [ ]:
run(["bash", "scripts/check_fixture_hashes.sh"])


## §2 Full federal run (P2 canonical demo)

Equivalent to:

```bash
uv run workbench validate run --config configs/german_credit_full.yaml
```


In [ ]:
run([
    "uv", "run", "workbench", "validate", "run",
    "--config", "configs/german_credit_full.yaml",
])
FULL_REPORT = ROOT / "artifacts" / "german-credit-demo-v1" / "german_credit_validation_report.md"
print("report:", FULL_REPORT, "exists=", FULL_REPORT.exists())


## §3 Audit chain


In [ ]:
run(["uv", "run", "workbench", "audit", "verify-chain"])


## §4 CA overlay run (P3)

Equivalent to:

```bash
uv run workbench validate run \
  --config configs/german_credit_ca_overlay.yaml \
  --output-dir artifacts/run_ca_overlay/
```


In [ ]:
run([
    "uv", "run", "workbench", "validate", "run",
    "--config", "configs/german_credit_ca_overlay.yaml",
    "--output-dir", "artifacts/run_ca_overlay/",
])
CA_REPORT = ROOT / "artifacts" / "run_ca_overlay" / "german_credit_validation_report.md"
print("report:", CA_REPORT, "exists=", CA_REPORT.exists())


## §5 P3 climax — side-by-side §II (Purpose and Scope)

Same model and German Credit rows; only `regulatory_frame.state_overlay` and report/audit framing change (R-10 §8).


In [ ]:
def extract_section_ii(md_text: str) -> str:
    m = re.search(
        r"(## II\. Purpose and Scope\n)(.*?)(?=\n## III\. |\Z)",
        md_text,
        flags=re.DOTALL,
    )
    if not m:
        raise ValueError("Section II not found in report")
    return m.group(1) + m.group(2).rstrip() + "\n"

baseline = extract_section_ii(FULL_REPORT.read_text(encoding="utf-8"))
overlay = extract_section_ii(CA_REPORT.read_text(encoding="utf-8"))

from IPython.display import Markdown, display

display(Markdown("### Baseline (`state_overlay: none`)"))
display(Markdown(baseline))
display(Markdown("---"))
display(Markdown("### CA overlay (`state_overlay: CA`)"))
display(Markdown(overlay))

# Compact structural diff of §II bullet lines
base_lines = [ln for ln in baseline.splitlines() if ln.strip().startswith("- ") or ln.strip().startswith("|")]
ca_lines = [ln for ln in overlay.splitlines() if ln.strip().startswith("- ") or ln.strip().startswith("|")]
only_base = [ln for ln in base_lines if ln not in ca_lines]
only_ca = [ln for ln in ca_lines if ln not in base_lines]
print("§II lines unique to baseline:", len(only_base))
for ln in only_base:
    print("  -", ln)
print("§II lines unique to CA overlay:", len(only_ca))
for ln in only_ca:
    print("  +", ln)


## §6 Limitations (German Credit)

- Non-US population; no race/ethnicity fields → **BISG not performed** (see notebook `02`).
- Drift on the monitor split is **injected covariate shift**, not temporal production drift.
- CA/CO overlays are **jurisdictional-frame demonstrations** only — not compliance determinations.

Continue to [`02_home_credit_bias_proxy.ipynb`](02_home_credit_bias_proxy.ipynb) for the P4 BISG architecture story.
